# SupportIQ — Stage 2.1: Schema Enforcement & Data Quarantine

> **Context:** End-to-End Fine-Tuning Pipeline & Cloud Deployments
> **Goal:** Build a strict Pydantic record validator. Any malformed row is routed to `data/interim/quarantine.jsonl` with an explicit reason—never silently dropped.
> Reference: `AGENT_GUIDE_v2.md` Subtask 2.1.


### 1. Why Schema Validation & Quarantine are Essential for Cloud Deployments
In a production fine-tuning pipeline:
- **No Silent Drops:** If corrupted data is silently dropped, data loss goes unnoticed.
- **Fail-Safe Training:** Malformed or empty strings during SFT QLoRA cause NaN losses and waste expensive GPU hours.
- **Unified Contract:** The Pydantic schemas defined here are the **exact same schemas** used later in the FastAPI serving container to validate user HTTP requests.


In [1]:
import json
from pathlib import Path
from typing import Any

from pydantic import BaseModel, Field, ValidationError, field_validator

from supportiq.data.load import load_raw_dataframe

print("Dependencies loaded successfully.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Dependencies loaded successfully.


### 2. Define the Pydantic Record Schema
Each row in the dataset must strictly satisfy:
- `flags`: non-empty string.
- `instruction`: string, length between 3 and 1,000 characters, not whitespace-only.
- `category`: uppercase/alphanumeric domain identifier.
- `intent`: snake_case customer intent identifier.
- `response`: string, length between 5 and 5,000 characters, not whitespace-only.


In [2]:
class CustomerSupportRecord(BaseModel):
    flags: str = Field(..., min_length=1, description="Dataset flag")
    instruction: str = Field(..., min_length=3, max_length=1000, description="Customer message")
    category: str = Field(..., min_length=2, max_length=64, description="High-level category")
    intent: str = Field(..., min_length=2, max_length=64, description="Specific intent")
    response: str = Field(..., min_length=5, max_length=5000, description="Assistant response")

    @field_validator("instruction", "response", mode="after")
    @classmethod
    def check_non_empty_whitespace(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("Field cannot be empty or pure whitespace.")
        return v

    @field_validator("category", mode="after")
    @classmethod
    def check_category_format(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("Category cannot be empty.")
        return v.strip().upper()

    @field_validator("intent", mode="after")
    @classmethod
    def check_intent_format(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("Intent cannot be empty.")
        return v.strip().lower()


print("CustomerSupportRecord schema compiled.")

CustomerSupportRecord schema compiled.


### 3. Load Raw Data
Load the preserved raw dataset from `data/raw/` using `load_raw_dataframe()`.


In [3]:
raw_df = load_raw_dataframe()
print(f"Loaded {raw_df.height:,} raw rows to validate.")

Loaded 26,872 raw rows to validate.


### 4. Validation Engine & Quarantine Logic
For every row:
- If valid -> add to `valid_records`.
- If invalid -> capture the exact error message and route to `quarantine_records`.


In [4]:
def validate_record_dict(
    record: dict[str, Any],
) -> tuple[bool, str | None, CustomerSupportRecord | None]:
    try:
        validated = CustomerSupportRecord(**record)
        return True, None, validated
    except ValidationError as e:
        error_msg = "; ".join(f"{err['loc'][0]}: {err['msg']}" for err in e.errors())
        return False, error_msg, None


print("Validation engine ready.")

Validation engine ready.


### 5. Validate the Complete Raw Dataset
Execute validation across all 26,872 rows and track results.


In [5]:
valid_records = []
quarantined_records = []

for idx, row in enumerate(raw_df.iter_rows(named=True)):
    is_valid, reason, model_obj = validate_record_dict(row)
    if is_valid and model_obj is not None:
        valid_records.append(model_obj.model_dump())
    else:
        quarantined_records.append(
            {"row_index": idx, "raw_record": row, "quarantine_reason": reason}
        )

print(f"Total processed:       {raw_df.height:,}")
print(
    f"Valid records:         {len(valid_records):,} ({len(valid_records) / raw_df.height * 100:.2f}%)"
)
print(
    f"Quarantined records:   {len(quarantined_records):,} ({len(quarantined_records) / raw_df.height * 100:.2f}%)"
)

Total processed:       26,872
Valid records:         26,872 (100.00%)
Quarantined records:   0 (0.00%)


### 6. Demonstrate Quarantine Behavior (Edge Case Testing)
To prove that our quarantine mechanism catches malformed inputs, we inject test edge cases:
1. Empty whitespace instruction.
2. Short response (< 5 chars).
3. Whitespace-only category.


In [6]:
synthetic_bad_rows = [
    {
        "flags": "B",
        "instruction": "   ",
        "category": "ORDER",
        "intent": "cancel_order",
        "response": "Valid response.",
    },
    {
        "flags": "B",
        "instruction": "Valid instruction",
        "category": "ORDER",
        "intent": "cancel_order",
        "response": "Hi",
    },
    {
        "flags": "",
        "instruction": "Valid instruction",
        "category": "ORDER",
        "intent": "cancel_order",
        "response": "Valid response.",
    },
]

test_quarantine = []
for bad_row in synthetic_bad_rows:
    is_valid, reason, _ = validate_record_dict(bad_row)
    test_quarantine.append({"record": bad_row, "is_valid": is_valid, "quarantine_reason": reason})

for item in test_quarantine:
    print(f"Input:    {item['record']}")
    print(f"Rejected: {not item['is_valid']}")
    print(f"Reason:   {item['quarantine_reason']}")
    print("-" * 60)

Input:    {'flags': 'B', 'instruction': '   ', 'category': 'ORDER', 'intent': 'cancel_order', 'response': 'Valid response.'}
Rejected: True
Reason:   instruction: Value error, Field cannot be empty or pure whitespace.
------------------------------------------------------------
Input:    {'flags': 'B', 'instruction': 'Valid instruction', 'category': 'ORDER', 'intent': 'cancel_order', 'response': 'Hi'}
Rejected: True
Reason:   response: String should have at least 5 characters
------------------------------------------------------------
Input:    {'flags': '', 'instruction': 'Valid instruction', 'category': 'ORDER', 'intent': 'cancel_order', 'response': 'Valid response.'}
Rejected: True
Reason:   flags: String should have at least 1 character
------------------------------------------------------------


### 7. Persist Quarantine Log (`data/interim/quarantine.jsonl`)
Write any quarantined rows to `data/interim/quarantine.jsonl` with full diagnostic lineage.


In [7]:
interim_dir = Path("../data/interim") if Path("../data/interim").exists() else Path("data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)
quarantine_path = interim_dir / "quarantine.jsonl"

with open(quarantine_path, "w", encoding="utf-8") as f:
    for q in quarantined_records:
        f.write(json.dumps(q) + "\n")

print(f"Quarantine log written to: {quarantine_path.resolve()}")
print(
    f"Quarantine file size: {quarantine_path.stat().st_size} bytes (records quarantined: {len(quarantined_records)})"
)

Quarantine log written to: /home/dvinix/Projects/supportiq/data/interim/quarantine.jsonl
Quarantine file size: 0 bytes (records quarantined: 0)


### 8. Stage 2.1 Findings & Extraction Plan
- **Dataset Conformance:** All 26,872 raw rows in the Bitext dataset strictly satisfy our Pydantic schema (0 malformed rows).
- **Quarantine Safety:** Simulated malformed rows were successfully caught and rejected with human-readable error reasons.
- **Next Subtask:** Extract the schema into `schemas/dataset.py` and the validation runner into `src/supportiq/data/validate.py`, then write unit tests in `tests/data/test_validate.py`.
